# Lightning AI Runner: Binary Grain-vs-Background Segmentation (no augmentation)

Lightning-AI runner for the **binary** task (`swin_binary_segmentation_221.py`): each pixel is
**grain (1)** vs **non-grain / background (0)**. Sibling of the 17-class runner; mirrors its
setup and fixes.

**Key points**
- **Latest labeling scheme:** the original 18-class masks (ids 0..17) are collapsed to binary.
  Scale bar (11) and watermark (17) are **ignored** (non-biological artifacts), background (0)
  is non-grain, everything else is grain (incl. sponge=16). This is baked into
  `multiclass_mask_to_binary`.
- **3-fold *stratified* CV:** folds are stratified over the *original* multiclass class presence
  so grain TYPES spread evenly across folds (`--cv_strategy stratified`, `--n_folds 3`).
- **Reuses the SSL backbone** from the 17-class runs (`ssl_swinv2_best.pth`) — no need to re-run SSL.
- Same Lightning plumbing as the 17-class runner: rclone + Studio-local disk, `--num_workers 0`
  (no dataloader deadlock), and the `pad_if_needed` crop fix for sub-512 images.

> Metrics are logged to **Weights & Biases** (one run per fold, grouped under one run name) and
> also written to `fold_<i>/val_metrics.csv` + `cv_summary.json`. Skip the W&B login cell (and
> drop the `--wandb_*` flags) to disable logging.

---

### One-time Studio setup (Lightning **terminal**, not this notebook)
```bash
curl https://rclone.org/install.sh | sudo bash        # install rclone
rclone config                                          # remote named exactly `gdrive` (scope 1, auto-config n)
cd ~ && git clone git@github.com:racyun/Payne_Lab_Carbon_Thin_Segmentation.git
rclone lsd gdrive:
```


## 0. Pull latest code from GitHub

In [ ]:
import os
REPO_ROOT = os.path.expanduser('~/Payne_Lab_Carbon_Thin_Segmentation')
os.chdir(REPO_ROOT)
os.environ['REPO_ROOT'] = REPO_ROOT
!git pull origin main
print('Repo:', REPO_ROOT)

## 1. Check runtime (GPU)

In [ ]:
import torch
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0),
          f'| VRAM {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    print('No GPU detected. Open a GPU Studio (Settings -> Compute) before training.')

## 2. Install Python dependencies

In [ ]:
!pip -q install --upgrade transformers tqdm wandb

## 2b. (Optional) Weights & Biases login

Skip this cell to disable W&B logging (also drop the `--wandb_*` flags in the finetune cell).

In [ ]:
import os
os.environ.setdefault('WANDB_PROJECT', 'payne-carbonate-segmentation')
import wandb
wandb.login()

## 3. Verify rclone is set up

In [ ]:
import subprocess
res = subprocess.run(['rclone', 'lsd', 'gdrive:'], capture_output=True, text=True)
if res.returncode != 0:
    raise RuntimeError('rclone not configured. Error:\n' + res.stderr)
print('rclone OK. Top-level Drive folders:')
print(res.stdout)

## 4. Config — Drive paths and Studio-local paths

In [ ]:
import os
from pathlib import Path

# Google Drive side (relative to the gdrive: remote root, i.e. 'My Drive')
DRIVE_ROOT     = 'Petrographic images_ML work'
LABELLED_DRIVE = DRIVE_ROOT + '/labelled images_PS/ALL_LABELS'   # no-aug binary uses ALL_LABELS
OUT_DRIVE      = DRIVE_ROOT + '/model_outputs_lightning'

# Studio-local disk (space-free paths)
LOCAL_ROOT         = Path('/teamspace/studios/this_studio/petro_data')
LABELED_IMG_LOCAL  = LOCAL_ROOT / 'labeled' / 'img'
LABELED_MASK_LOCAL = LOCAL_ROOT / 'labeled' / 'masks_machine'
SSL_CKPT           = LOCAL_ROOT / 'model_outputs' / 'ssl_full' / 'ssl_swinv2_best.pth'
OUT_BIN            = LOCAL_ROOT / 'model_outputs' / 'seg_binary_3fold'
for d in (LABELED_IMG_LOCAL, LABELED_MASK_LOCAL, OUT_BIN, SSL_CKPT.parent):
    d.mkdir(parents=True, exist_ok=True)

IMG_DIR  = str(LABELED_IMG_LOCAL)
MASK_DIR = str(LABELED_MASK_LOCAL)
os.environ['IMG_DIR']  = IMG_DIR
os.environ['MASK_DIR'] = MASK_DIR
os.environ['SSL_CKPT'] = str(SSL_CKPT)
os.environ['OUT_BIN']  = str(OUT_BIN)

print('Labeled  : gdrive:' + LABELLED_DRIVE)
print('SSL ckpt :', SSL_CKPT)
print('Output   :', OUT_BIN)

## 5. Sync labeled data + SSL backbone from Drive

`rclone copy` only transfers missing/changed files. The SSL backbone is reused from the
17-class runs; if it isn't on local disk it's pulled from `model_outputs_221/ssl_full`.

In [ ]:
import subprocess, os
from pathlib import Path

def rclone_copy(remote_rel, local_path):
    Path(local_path).mkdir(parents=True, exist_ok=True)
    subprocess.run(['rclone', 'copy', 'gdrive:' + remote_rel, str(local_path),
                    '--progress', '--transfers=8', '--checkers=16'], check=True)

rclone_copy(LABELLED_DRIVE + '/img', LABELED_IMG_LOCAL)
rclone_copy(LABELLED_DRIVE + '/masks_machine', LABELED_MASK_LOCAL)

if not SSL_CKPT.exists():
    print('SSL backbone not on local disk; pulling from Drive...')
    rclone_copy(DRIVE_ROOT + '/model_outputs_221/ssl_full/ssl_swinv2_best.pth', SSL_CKPT.parent)

print('labeled images:', len(list(LABELED_IMG_LOCAL.glob('*'))),
      '| masks:', len(list(LABELED_MASK_LOCAL.glob('*'))))
print('SSL backbone present:', SSL_CKPT.exists())

## 6. Smoke test (dataloader only)

In [ ]:
import os
os.chdir(REPO_ROOT)
!python -u code/model_training_pipeline/swin_binary_segmentation_221.py \
  --img_dir "$IMG_DIR" --mask_dir "$MASK_DIR" --no_train

## 7. Binary finetune — 3-fold stratified CV

Grain vs background, SSL-initialized backbone, 50 epochs, `--num_workers 0`. Writes per-fold
checkpoints `fold_<i>/best_upernet_swinv2_binary.pth`, per-fold `val_metrics.csv`, and an
aggregate `cv_summary.json`. Each fold logs to W&B as `seg_no_aug_binary_3fold_fold<i>`, all
grouped under `seg_no_aug_binary_3fold`. (Drop `--backbone_checkpoint` to train from the
ImageNet backbone; drop the `--wandb_*` flags to disable logging.)

In [ ]:
import os
os.chdir(REPO_ROOT)
os.environ['WANDB_DIR'] = str(OUT_BIN / 'wandb'); os.makedirs(os.environ['WANDB_DIR'], exist_ok=True)
!python -u code/model_training_pipeline/swin_binary_segmentation_221.py \
  --img_dir "$IMG_DIR" \
  --mask_dir "$MASK_DIR" \
  --n_folds 3 \
  --cv_strategy stratified \
  --epochs 50 \
  --batch_size 2 \
  --crop 512 \
  --num_workers 0 \
  --lr 3e-4 \
  --backbone_checkpoint "$SSL_CKPT" \
  --output_dir "$OUT_BIN" \
  --wandb_project payne-carbonate-segmentation \
  --wandb_run_name seg_no_aug_binary_3fold

## 8. Sync outputs back to Google Drive

In [ ]:
import subprocess
subprocess.run(['rclone', 'copy', str(OUT_BIN), 'gdrive:' + OUT_DRIVE + '/seg_binary_3fold',
                '--progress', '--transfers=8', '--checkers=16'], check=True)
print('Outputs synced to gdrive:' + OUT_DRIVE + '/seg_binary_3fold')

## 9. Binary confusion matrix (2x2) on a chosen fold

Loads a fold's binary checkpoint and reproduces the SAME stratified split it validated on
(stratified over the original multiclass presence). Set `FOLD`/`N_FOLDS`/`CV_STRATEGY` to
match the finetune run.

In [ ]:
import os, sys
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
import matplotlib.pyplot as plt
from torchvision.transforms.v2 import CenterCrop, Compose

PIPE_DIR = Path(REPO_ROOT) / 'code' / 'model_training_pipeline'
if str(PIPE_DIR) not in sys.path:
    sys.path.insert(0, str(PIPE_DIR))
from swin_binary_segmentation_221 import (
    BinaryCarbonateDataset, get_binary_model, NUM_BINARY_CLASSES, BINARY_CLASS_NAMES, IGNORE_INDEX,
)
from swin_training_pipeline_221 import (
    NUM_CLASSES, ARTIFACT_CLASS_IDS, build_class_presence_matrix,
    stratified_kfold_indices, kfold_train_val_indices, confusion_matrix as cm_fn,
)

# --- Config: MUST match the finetune run (cell 7) ---
OUT_BIN     = Path('/teamspace/studios/this_studio/petro_data/model_outputs/seg_binary_3fold')
FOLD        = 0
N_FOLDS     = 3
CV_STRATEGY = 'stratified'
SEED        = 1337
CROP        = 512
CKPT        = OUT_BIN / f'fold_{FOLD}' / 'best_upernet_swinv2_binary.pth'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# --- Reproduce the exact validation split fold FOLD used (stratified over multiclass presence) ---
probe = BinaryCarbonateDataset(root='.', img_dir=IMG_DIR, mask_dir=MASK_DIR,
                               transforms=None, normalize=True, strict=True)
n = len(probe)
if CV_STRATEGY == 'stratified':
    presence = build_class_presence_matrix(probe.pairs, NUM_CLASSES, IGNORE_INDEX, ARTIFACT_CLASS_IDS)
    splits = stratified_kfold_indices(presence, N_FOLDS, SEED)
else:
    splits = kfold_train_val_indices(n, N_FOLDS, SEED)
val_idx = list(np.asarray(splits[FOLD][1]).tolist())

val_full = BinaryCarbonateDataset(root='.', img_dir=IMG_DIR, mask_dir=MASK_DIR,
                                  transforms=Compose([CenterCrop((CROP, CROP))]),
                                  normalize=True, strict=False, print_pair_count=False)
val_ds = Subset(val_full, val_idx)
val_loader = DataLoader(val_ds, batch_size=1, shuffle=False, num_workers=0,
                        pin_memory=torch.cuda.is_available())
print(f'Fold {FOLD}/{N_FOLDS} | val samples: {len(val_ds)} | ckpt: {CKPT}')

model = get_binary_model(IGNORE_INDEX).to(device)
model.load_state_dict(torch.load(str(CKPT), map_location=device, weights_only=False)['model_state'])
model.eval()

cm_total = torch.zeros(NUM_BINARY_CLASSES, NUM_BINARY_CLASSES, dtype=torch.float64)
with torch.no_grad():
    for imgs, labels in val_loader:
        imgs = imgs.to(device); labels = labels.to(device)
        logits = model(pixel_values=imgs).logits
        logits = F.interpolate(logits, size=labels.shape[-2:], mode='bilinear', align_corners=False)
        preds = logits.argmax(dim=1)
        cm_total += cm_fn(preds, labels, NUM_BINARY_CLASSES, IGNORE_INDEX).cpu().double()

cm_np = cm_total.numpy()
out_dir = OUT_BIN / f'fold_{FOLD}'
np.save(out_dir / 'cm_binary_raw.npy', cm_np)

row = cm_np / np.maximum(cm_np.sum(1, keepdims=True), 1)
fig, ax = plt.subplots(figsize=(4.5, 4))
im = ax.imshow(row, vmin=0, vmax=1, cmap='Blues')
ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
ax.set_xticklabels(BINARY_CLASS_NAMES); ax.set_yticklabels(BINARY_CLASS_NAMES)
ax.set_xlabel('Predicted'); ax.set_ylabel('Ground truth')
ax.set_title(f'Fold {FOLD} binary CM (row-norm; diag = recall)')
for i in range(2):
    for j in range(2):
        ax.text(j, i, f'{row[i, j]:.3f}', ha='center', va='center',
                color='white' if row[i, j] > 0.5 else 'black')
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout(); plt.savefig(out_dir / 'cm_binary_row_normalized.png', dpi=150, bbox_inches='tight'); plt.show()
print('Saved', out_dir / 'cm_binary_row_normalized.png')

diag = np.diag(cm_np)
recall    = diag / np.maximum(cm_np.sum(1), 1)
precision = diag / np.maximum(cm_np.sum(0), 1)
union     = cm_np.sum(1) + cm_np.sum(0) - diag
iou       = np.where(union > 0, diag / np.maximum(union, 1), np.nan)
print(f"\nFold {FOLD} per-class metrics:")
print(f"{'class':<12}{'recall':>9}{'precision':>11}{'IoU':>8}")
for i, name in enumerate(BINARY_CLASS_NAMES):
    print(f"{name:<12}{recall[i]:>9.3f}{precision[i]:>11.3f}{iou[i]:>8.3f}")